# TFT model


1. Imports
2. Load dataset
3. Clean dataset
4. Create timeseries structure
5. Create TFT dataset
6. Create model
7. Train model
8. Monitor training
9. Generate predictions
10. Signal detection
11. Save model

## Some boring shit before we get started
### Environment Setup for the TFT Notebook

The following steps describe how the Python environment used for this notebook was created.
This ensures reproducibility of the Temporal Fusion Transformer (TFT) experiments.

#### 1. Install Python 3.11

Install **Python 3.11 (64-bit)**.
Newer versions (e.g., 3.13) may cause compatibility problems with PyTorch and related libraries.

---

#### 2. Create a virtual environment

Open **PowerShell** in the project directory and run:

```
py -3.11 -m venv tft_env
```

---

#### 3. Activate the virtual environment

```
.\tft_env\Scripts\activate
```

---

#### 4. Upgrade pip

```
python -m pip install --upgrade pip
```

---

#### 5. Install the core libraries

First install PyTorch:

```
pip install torch torchvision torchaudio
```

Then install the remaining dependencies:

```
pip install pytorch-lightning pytorch-forecasting torchmetrics pandas numpy matplotlib scikit-learn jupyterlab
```

---

#### 6. Create a dedicated Jupyter kernel

Install the kernel package (**It's like creating a personal kernel for a specific enviroment, isolating it from eventual bugs from other directory files**):

```
pip install ipykernel
```

Register the environment as a Jupyter kernel:

```
python -m ipykernel install --user --name tft_env
```

---

#### 7. Start JupyterLab

```
jupyter lab
```

Inside JupyterLab, select the kernel from 'Kernel' upside:

```
tft_env
```

---

#### 8. Verify the installation

Run the following code to confirm that the environment is correctly configured:

```python
import torch
import pytorch_forecasting
import pytorch_lightning

print("torch:", torch.__version__)
print("forecasting:", pytorch_forecasting.__version__)
print("lightning:", pytorch_lightning.__version__)
print("GPU:", torch.cuda.is_available())
```

Expected output example:

```
torch: 2.11.0+cpu
forecasting: 1.6.1
lightning: 2.6.1
GPU: False
```

This confirms that the environment is properly configured and ready for the TFT pipeline.


# Now we start

## 1. Imports

In [1]:
import sys

# garantir ambiente correto
print(sys.executable)

# instalar tudo no kernel certo
!{sys.executable} -m pip install torch pytorch-lightning pytorch-forecasting torchmetrics

C:\projects\Libellula\notebooks\Predictive_TFT_model\tft_env\Scripts\python.exe



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\projects\Libellula\notebooks\Predictive_TFT_model\tft_env\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import torch
import pytorch_forecasting
import pytorch_lightning

print("torch:", torch.__version__)
print("forecasting:", pytorch_forecasting.__version__)
print("lightning:", pytorch_lightning.__version__)
print("GPU:", torch.cuda.is_available())

C:\projects\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


torch: 2.11.0+cpu
forecasting: 1.7.0
lightning: 2.6.1
GPU: False


In [3]:
import pandas as pd
import numpy as np
import torch

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
'''
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import LearningRateMonitor
'''

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.callbacks import LearningRateMonitor

import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load datasets

In [ ]:
DATA_PATH = "../../data/processed/financial_tools_datset.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
print(df.shape)
print(df.columns)
df.info()

## 3. Cleaning dataset

In [ ]:
# inverter ordem temporal
df = df.iloc[::-1].reset_index(drop=True)

# converter data
df["Date"] = pd.to_datetime(df["Date"])

# criar índice temporal
df["time_idx"] = np.arange(len(df))

# renomear colunas problemáticas
df = df.rename(columns={
    "%K10":"K10",
    "%D10":"D10",
    "%K30":"K30",
    "%D30":"D30",
    "%K200":"K200",
    "%D200":"D200"
})

# remover Change %
df = df.drop(columns=["Change %"])

print(df.head())
print("colunas:", df.columns)

## 4. Creating timeseries structure
### 4.1 Creating targets

In [ ]:
df["target"] = df["Price"].shift(-1)

df = df.dropna()

print(df.tail())

### 4.2 Creating series ID
TFT needs an identifier:

In [ ]:
df["series"] = "EURUSD"

## 5. Creating TFT dataset

### 5.1 Temporal parameters

In [ ]:
max_encoder_length = 64
max_prediction_length = 1

### 5.2 Creating TimeSeriesDataset

In [ ]:
training = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="target",
    group_ids=["series"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=[
        "time_idx"
    ],

    time_varying_unknown_reals=[
        'Price','Open','High','Low',
        'short_mavg','long_mavg','signal',
        'EMA10','EMA30','EMA200',
        'ROC10','ROC30',
        'MOM10','MOM30',
        'RSI10','RSI30','RSI200',
        'K10','D10','K30','D30','K200','D200'
    ]
)

### 5.3 Creating TimeSeriesDataset (validation)

In [ ]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    df,
    predict=True,
    stop_randomization=True
)

val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=64,
    num_workers=0
)

### 5.4 Dataloader

In [ ]:
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=64,
    num_workers=0,
    shuffle=False
)

## 6. Create model

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

### 6.1 Callbacks

In [ ]:
early_stop = EarlyStopping(
    monitor="train_loss",
    patience=5,
    mode="min"
)

lr_logger = LearningRateMonitor()

### 6.2 Trainer

In [ ]:
trainer = pl.Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_logger]
)

### 6.3 Training the model

In [ ]:
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

## 7. Monitoring training

In [ ]:
metrics = pd.DataFrame(trainer.callback_metrics, index=[0])

print(metrics)

## 8. Generating predictions

In [ ]:
raw_predictions = tft.predict(train_dataloader, mode="raw", return_x=True)

print(raw_predictions)

### 8.1 Editing raw output and comparing prediction to real data

In [ ]:
import pandas as pd 
pred = raw_predictions.output.prediction.detach().cpu().numpy() 
pred_median = pred[:,0,3] 
real = raw_predictions.x["decoder_target"][:,0].detach().cpu().numpy() 
compare = pd.DataFrame({ "real": real, "prediction": pred_median }) 
print(compare.head())

### 8.2 Getting model MAE analysis performance 

In [ ]:
import numpy as np

mae = np.mean(np.abs(compare["real"] - compare["prediction"]))
print("MAE:", mae)

## 9. Signal detection
### 9.1 Using quantile to establish a minimum treeshold for signal generation

In [ ]:
import numpy as np

pred = raw_predictions.output.prediction.detach().cpu().numpy()

q30 = pred[:,0,2]
q50 = pred[:,0,3]
q70 = pred[:,0,4]

confidence = 1 / (1 + np.abs(q70 - q30))

signal_df = pd.DataFrame({
    "prediction": q50,
    "confidence": confidence
})

threshold = 0.6

signal_df["signal"] = signal_df["confidence"] > threshold

print(signal_df.head())

## 10. Saving model

In [ ]:
torch.save(tft.state_dict(), "tft_model.pth")